In [1]:
import pandas as pd
import numpy as np
import random
import re
from itertools import product, combinations
import os
import pickle
from collections import defaultdict, Counter
import warnings
import json
import copy
from itertools import cycle
import math
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from scipy.linalg import fractional_matrix_power
from IPython.display import display
from pandas.api.types import is_numeric_dtype
import importlib

from sklearn import metrics, set_config
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, precision_score, average_precision_score, recall_score, accuracy_score, f1_score, confusion_matrix, roc_curve, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

import torch
import torch.nn as nn 

import custom_transformers; importlib.reload(custom_transformers)
import jp_utils; importlib.reload(jp_utils)
from jp_utils import get_new_cols, drop_useless_cols, load_data, check_filename, write_file, get_hgnc26, set_idx, limit_df
from custom_transformers import get_current_cols, RemoveNearlyConstant, MiniCorrelationAnalysis, FilterVariance, WeightedUnivar, SelectFeatures, RemoveRarelyMutated, RemoveLowExpression, CrossDatasetExpressionSurvival

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

import data_prep; importlib.reload(data_prep)
from data_prep import TCGA, TRACERx, LUNG, prepare_data, Table1, MutationFrequencies, AlignTRACERx, SelectScalers

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 150)

# https://xenabrowser.net/datapages/?cohort=TCGA%20Lung%20Adenocarcinoma%20(LUAD)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443 
# https://gdc.cancer.gov/about-data/publications/pancanatlas
paths = {
    'luad_pc': '../data/luad_pancan/', 'luad_fh': '../data/luad_fh/', 'luad_xena': '../data/luad_xena/', 
    'combined_xena': '../data/combined_xena/', 'gdc': '../data/gdc/', 'tracerx': '../data/tracerx/', 'models': 'models/'
}
data_path = '../data/'
keep = {
        'luad_fh': { # 3
            'data_clinical_patient': ['patient_id', 'sex', 'race', 'ethnicity', 'primary_site_patient', 'initial_pathologic_dx_year', 'residual_tumor', 'ajcc_staging_edition', 'ajcc_tumor_pathologic_pt', 'ajcc_nodes_pathologic_pn', 'ajcc_metastasis_pathologic_pm', 'ajcc_pathologic_tumor_stage', 'tobacco_smoking_history_indicator', 'smoking_year_started', 'smoking_year_stopped', 'smoking_pack_years', 'ecog_score', 'new_tumor_event_after_initial_treatment', 'age', 'dfs_status', 'dfs_months', 'treatment_outcome_first_course'],
            'data_clinical_sample': ['patient_id', 'sample_id', 'sample_type']
        },
        'luad_pc': { # 2
            'data_clinical_patient': ['patient_id', 'age', 'sex', 'ajcc_pathologic_tumor_stage', 'ajcc_staging_edition', 'days_to_birth', 'ethnicity', 'path_m_stage', 'path_n_stage', 'path_t_stage', 'race', 'dfs_status', 'dfs_months'],
            'data_clinical_sample': ['patient_id', 'sample_id', 'aneuploidy_score', 'msi_score_mantis', 'msi_sensor_score', 'tbl_score', 'tumor_type']
        },
        'luad_xena': { # 1
            'phenotypes': ['sampleid', 'age_at_initial_pathologic_diagnosis', 'anatomic_neoplasm_subdivision', 'days_to_additional_surgery_locoregional_procedure', 'days_to_additional_surgery_metastatic_procedure', 'days_to_birth', 'days_to_new_tumor_event_after_initial_treatment', 'gender', 'lost_follow_up', 'new_neoplasm_event_type', 'number_pack_years_smoked', 'pathologic_m', 'pathologic_n', 'pathologic_t', 'pathologic_stage', 'residual_tumor', 'sample_type', 'stopped_smoking_year', 'system_version', 'tobacco_smoking_history', 'year_of_tobacco_smoking_onset', 'tobacco_smoking_history_indicator', 'year_of_initial_pathologic_diagnosis', 'primary_therapy_outcome_success']
        },
        'combined_xena': { # 7
            'phenotypes': ['sampleid', 'pathology', 'pathology_updated', '_primary_disease', 'age_at_initial_pathologic_diagnosis', 'anatomic_neoplasm_subdivision', 'days_to_additional_surgery_locoregional_procedure', 'days_to_additional_surgery_metastatic_procedure', 'days_to_birth', 'days_to_new_tumor_event_after_initial_treatment', 'gender', 'histological_type', 'lost_follow_up', 'new_neoplasm_event_type', 'number_pack_years_smoked', 'pathologic_m', 'pathologic_n', 'pathologic_t', 'pathologic_stage', 'residual_tumor', 'sample_type', 'stopped_smoking_year', 'system_version', 'tobacco_smoking_history', 'year_of_tobacco_smoking_onset', 'tobacco_smoking_history_indicator', 'year_of_initial_pathologic_diagnosis', 'primary_therapy_outcome_success']
        },
        'gdc': { # luad_gdc_phenotypes 6; cdr 4; followup 5 
            'luad_gdc_phenotypes': ['sample', 'cigarettes_per_day.exposures', 'years_smoked.exposures', 'pack_years_smoked.exposures', 'race.demographic', 'gender.demographic', 'ethnicity.demographic', 'vital_status.demographic', 'age_at_index.demographic', 'days_to_birth.demographic', 'synchronous_malignancy.diagnoses', 'ajcc_pathologic_stage.diagnoses', 'age_at_diagnosis.diagnoses', 'year_of_diagnosis.diagnoses', 'primary_diagnosis.diagnoses', 'ajcc_staging_system_edition.diagnoses', 'ajcc_pathologic_t.diagnoses', 'ajcc_pathologic_n.diagnoses', 'ajcc_pathologic_m.diagnoses','tumor_descriptor.samples', 'sample_type.samples'],
            'cdr': ['bcr_patient_barcode', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'initial_pathologic_dx_year', 'birth_days_to', 'new_tumor_event_type', 'new_tumor_event_dx_days_to', 'dfi', 'dfi.time', 'type', 'vital_status', 'residual_tumor', 'histological_type', 'treatment_outcome_first_course'],
            'followup': ['bcr_patient_barcode', 'gender', 'days_to_birth', 'age_at_initial_pathologic_diagnosis', 'histological_type', 'race', 'ethnicity', 'year_of_initial_pathologic_diagnosis', 'pathologic_t', 'pathologic_m', 'pathologic_n', 'system_version', 'pathologic_stage', 'anatomic_neoplasm_subdivision', 'residual_tumor', 'tobacco_smoking_history', 'number_pack_years_smoked', 'stopped_smoking_year', 'year_of_tobacco_smoking_onset', 'days_to_new_tumor_event_after_initial_treatment', 'new_neoplasm_event_type', 'primary_therapy_outcome_success']
        },
        'tracerx': {
            'clinicohistopathological_data': ['sample_name_cruk', 'tumour_id_muttable_cruk', 'cruk_id', 'is.primary.tumour.tx421', 'histology_per_tumour_id_muttable', 'smoking_status_merged', 'pathologytnm'],
            'TRACERx421_all_patient_df': ['cruk_id', 'tumour_id_muttable_cruk', 'tumour_id_per_patient', 'age', 'sex', 'ethnicity', 'cigs_perday', 'years_smoking', 'packyears', 'smoking_status_merged', 'ecog_ps', 'pathologytnm', 'pt_stage_per_patient', 'pn_stage_per_patient', 'margin_status_per_patient', 'histology_lesion1_merged', 'histology_multi_full', 'recurrence_time_use', 'first_dfs_any_event_rec.or.new.primary', 'first_event_during_followup', 'cens_dfs', 'dfs_time'],
            'TRACERx421_all_tumour_df': ['cruk_id', 'tumour_id_muttable_cruk', 'clinical_sex', 'age', 'ethnicity', 'histology_3', 'smoking_status_merged', 'cigs_perday', 'years_smoking', 'pack_years', 'pathologytnm'],
            'sampleOverview': ['patient_id', 'tumour_id', 'region', 'sampletype', 'sampletypedetail']
        }
    }

In [2]:
hgnc = get_hgnc26()
lung1, tcga1, tracerx1 = prepare_data(paths, keep, hgnc, mut_transform='binarize')
lung1.preprocess(presplit=True, fn='tcga_split_mar2', test_size=0.15, valid_size=0.15)
lung1.finalize()

---- TCGA ----
Expression shape: (510, 17375), Mutation shape: (562, 16184)
All have clinical (otherwise no label):   Expr + Mut = 132   Expr only = 1   Mut only = 2   Clin only = 3
Cutoff used: Median TTR (434.0)
132 patients (66 late, 66 early)

---- TracerX ----
Cutoff used: Predetermined (434.0)
66 patients (30 late, 36 early)

TCGA splits: 92 train, 20 valid, 20 test

Features removed: 0 clin (7 remaining) 39 expr (17086 remaining) 1318 mut (8369 remaining)


In [3]:
def get_mut_distribution(lung): # for plotting variant distribution
    mut_incl = [gene.replace('_mut','') for gene in lung.og_cols['mut']]
    tcga_raw_mut = lung.tcga.raw_mut[lung.tcga.raw_mut['approved_symbol'].isin(mut_incl)].copy()
    tcga_mut_dist = tcga_raw_mut[['Tumor_Sample_Barcode','Variant_Classification']].merge(lung.tcga.clin['label'], left_on='Tumor_Sample_Barcode', right_index=True).groupby(['Variant_Classification','label']).size().unstack(fill_value=0)
    trcr_raw_mut = lung.trcr.raw_mut[lung.trcr.raw_mut['approved_symbol'].isin(mut_incl)].copy()
    trcr_raw_mut['exonic.func'] = trcr_raw_mut['exonic.func'].replace({'unknown': 'exonic - unknown', 'UNKNOWN': 'exonic - unknown'})
    temp = trcr_raw_mut[['exonic.func', 'func']].bfill(axis=1).iloc[:, 0]
    trcr_raw_mut['exonic.func'] = temp
    trcr_mut_dist = trcr_raw_mut[['tumour_id','exonic.func']].merge(lung.trcr.clin['label'], left_on='tumour_id', right_index=True).groupby(['exonic.func','label']).size().unstack(fill_value=0)
    return tcga_mut_dist, trcr_mut_dist

def plot_variant_distribution(name, grouped_data, size=(11, 6.5), dpi=800, bar_height=0.36):
    fig, ax = plt.subplots(figsize=size, dpi=dpi)
    labels = [i.replace('_',' ').replace(' Ins', ' Insertion').replace(' Del', ' Deletion') for i in grouped_data.index.tolist()]
    y = np.arange(len(labels)) # Label locations
    rects1 = ax.barh(y + bar_height/2, grouped_data.iloc[:, 0], bar_height, label='Late recurrence', color='#B2B2B2')
    rects2 = ax.barh(y - bar_height/2, grouped_data.iloc[:, 1], bar_height, label='Early recurrence', color='#777777')
    ax.set_xlabel('Variant count', fontsize=10.5)
    ax.set_title(f'{name} Distribution of Variant Classification Types by Recurrence Label', fontweight='bold', fontsize=12)
    ax.set_yticks(y)
    ax.tick_params(axis='y', pad=5, reset=True, color='black', labelsize=10.5, right=False, length=0)
    ax.set_yticklabels(labels, fontsize=10.5)
    ax.legend(prop={'weight': 'roman', 'size': 10.5})
    ax.bar_label(rects1, label_type='edge', padding=3, fontsize=10.5, fontweight='roman')
    ax.bar_label(rects2, label_type='edge', padding=3, fontsize=10.5, fontweight='roman')
    ax.margins(y=0.02, x=0.057)
    plt.tight_layout()
    plt.show()

tcga_md, trcr_md = get_mut_distribution(lung1)

df1 = lung1.tcga.raw_mut[['Consequence', 'Variant_Classification', 'Variant_Type']].value_counts(dropna=False)
df2 = lung1.trcr.raw_mut[['func', 'exonic.func']].value_counts(dropna=False)
write_file('csv', 'tcga_variant_counts', df1, '../results/')
write_file('csv', 'trcr_variant_counts', df2, '../results/')

In [4]:
# The core problem: TCGA uses VEP/Ensembl via the GDC MAF pipeline (giving Variant_Classification + Variant_Type), while TRACERx uses ANNOVAR with its own func/exonic.func labels.
class StandardizeVariants:
    def __init__(self, lung, stdcol='Variant Category'):
        mut_incl = [gene.replace('_mut','') for gene in lung.og_cols['mut']]
        self.tcga_raw = lung.tcga.raw_mut.loc[lung.tcga.raw_mut['approved_symbol'].isin(mut_incl)].copy()
        self.trcr_raw = lung.trcr.raw_mut.loc[lung.trcr.raw_mut['approved_symbol'].isin(mut_incl)].copy()
        self.stdcol = stdcol
        self.get_tcga_maps(); self.get_trcr_maps()
        self.tcga_rc = self.reclassify_tcga()
        self.trcr_rc = self.reclassify_trcr()

    def get_tcga_maps(self): # TCGA: Variant_Classification → Standardized
        self.tcga_vc_map = {
            'Missense_Mutation':      'Missense',
            'Nonsense_Mutation':      'Nonsense',
            'Nonstop_Mutation':       'Stop Loss',
            'Translation_Start_Site': 'Translation Start Site',
            'Frame_Shift_Del':        'Frameshift Deletion',
            'Frame_Shift_Ins':        'Frameshift Insertion',
            'In_Frame_Del':           'In-frame Deletion',
            'In_Frame_Ins':           'In-frame Insertion',
            'Splice_Site':            'Splice Site',
            'Splice_Region':          'Splice Region',
            'Intron':                 'Intronic',
            "3'UTR":                  "3'UTR",
            "5'UTR":                  "5'UTR",
            "3'Flank":                "3'Flank",
            "5'Flank":                "5'Flank"
        }

    def get_trcr_maps(self): # TRACERx: func + exonic.func → Standardized ()
        self.trcr_exonic_map = { # ANNOVAR produces 'func': Func.refGene, genomic region
            'nonsynonymous SNV':          'Missense',
            'stopgain SNV':               'Nonsense',
            'stoploss SNV':               'Stop Loss',
            'startloss SNV':              'Translation Start Site', # this is the equivalent but no instances in TRACERx
            'frameshift insertion':       'Frameshift Insertion',
            'frameshift deletion':        'Frameshift Deletion', # This is the equivalent but no instances in TRACERx
            'nonframeshift insertion':    'In-frame Insertion',
            'nonframeshift deletion':     'In-frame Deletion', # This is the equivalent but no instances in TRACERx
            # Block substitutions: multi-nucleotide complex indels (directionality ambiguous)
            'frameshift substitution':    'Frameshift InDel',
            'nonframeshift substitution': 'In-frame InDel', 
            'nonsynonymous':              'Missense', # all dinucleotide
            'immediate-stopgain':         'Nonsense', # ⚠ see note 
            'unknown':                    'Unclassified',
            'UNKNOWN':                    'Unclassified'
        }
        self.trcr_func_map = { # ANNOVAR produces 'exonic.func': ExonicFunc.refGene, exonic consequence (only populated when func == 'exonic')
            'intronic':            'Intronic',
            'UTR3':                "3'UTR",
            'UTR5':                "5'UTR",
            'intergenic':          'Intergenic',
            'splicing':            'Splice Site',
            'upstream':            "5'Flank",
            'downstream':          "3'Flank",
            'upstream;downstream': "5'Flank" # truncated to the first element (this is what maftools does, only 1 instance I checked manually)
        }
    
    def reclassify_tcga(self):
        def classify_row(row):
            vc = row['Variant_Classification']
            cat = self.tcga_vc_map.get(vc)
            if cat is None: 
                warnings.warn(f'Unmapped TCGA Variant_Classification: {vc}', stacklevel=2)
                return 'Unclassified'
            return cat
        df = self.tcga_raw.copy()
        df[self.stdcol] = df.apply(classify_row, axis=1)
        return df
    
    def reclassify_trcr(self):
        def classify_row(row):
            func, exonic_func = row['func'], row['exonic.func']
            if func == 'exonic': # Exonic variants: use the finer-grained exonic.func label
                if pd.notna(exonic_func) and exonic_func != "":
                    cat = self.trcr_exonic_map.get(exonic_func)
                    if cat is None:
                        warnings.warn(f'Unmapped TRACERx exonic.func: {exonic_func}', stacklevel=2)
                        return 'Unclassified'
                    return cat
                warnings.warn("Found func=='exonic' with missing exonic.func value", stacklevel=2) # exonic but no exonic.func label
                return 'Unclassified'
            cat = self.trcr_func_map.get(func) # Non-exonic variants: use func directly
            if cat is None:
                warnings.warn(f'Unmapped TRACERx func: {func}', stacklevel=2)
                return 'Unclassified'
            return cat
        df = self.trcr_raw.copy()
        df[self.stdcol] = df.apply(classify_row, axis=1)
        return df

    def summarize(self, df): # count + percentage summary table sorted by count descending
        counts = df[self.stdcol].value_counts().rename_axis(self.stdcol).reset_index(name='n')
        counts['pct'] = (counts['n'] / counts['n'].sum() * 100).round(2)
        return counts
    
    def compare_datasets(self, combine_n_pct=False): # produce a side-by-side comparison table of variant category frequencies
        tcga_s = self.summarize(self.tcga_rc).rename(columns={'n': 'TCGA_n', 'pct': 'TCGA_pct'})
        trcr_s = self.summarize(self.trcr_rc).rename(columns={'n': 'TRACERx_n', 'pct': 'TRACERx_pct'})
        merged = pd.merge(tcga_s, trcr_s, on=self.stdcol, how='outer').fillna(0)
        merged[['TCGA_n', 'TRACERx_n']] = merged[['TCGA_n', 'TRACERx_n']].astype(int)
        merged = merged.sort_values('TCGA_n', ascending=False).reset_index(drop=True)
        if combine_n_pct:
            merged['TCGA [N (%)]'] = merged.apply(lambda r: f"{r['TCGA_n']:,} ({r['TCGA_pct']:.2f})", axis=1)
            merged['TRACERx [N (%)]'] = merged.apply(lambda r: f"{r['TRACERx_n']:,} ({r['TRACERx_pct']:.2f})", axis=1)
            merged = merged[[self.stdcol, 'TCGA [N (%)]', 'TRACERx [N (%)]']]
        return merged

sv = StandardizeVariants(lung1)
comparison = sv.compare_datasets(combine_n_pct=True)

In [5]:
sv.trcr_raw[sv.trcr_raw['func']=='upstream;downstream']

,mutation_id,patient_id,tumour_id,chr,start,stop,ref,var,func,exonic.func,NucleotideChange,AAChange,GL_VAF,GL_nAlt,GL_depth,ITHState,RegionSum,Is.present,PyCloneClonal_SC,PyCloneCluster_SC,cleanCluster_SC,PyCloneClonal_SC_TB_corrected,PyCloneCluster_SC_TB_corrected,PhyloCCF_SC,combTiming_SC,MutCPN_SC,MajorCPN_SC,MinorCPN_SC,DriverMut,approved_symbol
32849,CRUK0418:1:149858278:T,CRUK0418,CRUK0418,chr1,149858278,149858278,T,A,upstream;downstream,NaN,NaN,NaN,0.0,0,177,3,SU_T1.R1:0/218;SU_T1.R2:0/201;SU_T1.R3:0/197;S...,SU_T1.R1:FALSE;SU_T1.R2:FALSE;SU_T1.R3:FALSE;S...,S,3.0,True,S,3.0,SU_T1.R1:0;SU_T1.R2:0;SU_T1.R3:0;SU_T1.R4:0;SU...,subclonal,SU_T1.R1:0;SU_T1.R2:0;SU_T1.R3:0;SU_T1.R4:0;SU...,SU_T1.R1:4;SU_T1.R2:4;SU_T1.R3:4;SU_T1.R4:4;SU...,SU_T1.R1:2;SU_T1.R2:2;SU_T1.R3:2;SU_T1.R4:2;SU...,False,H2AC20


In [6]:
comparison

,Variant Category,TCGA [N (%)],TRACERx [N (%)]
0,Missense,"29,356 (76.75)","17,240 (43.48)"
1,Nonsense,"2,423 (6.33)","1,259 (3.18)"
2,3'UTR,"1,648 (4.31)",942 (2.38)
3,Splice Site,"1,049 (2.74)",542 (1.37)
4,5'UTR,951 (2.49),919 (2.32)
5,Frameshift Deletion,932 (2.44),0 (0.00)
6,Intronic,800 (2.09),"16,665 (42.03)"
7,Splice Region,486 (1.27),0 (0.00)
8,Frameshift Insertion,239 (0.62),196 (0.49)
9,In-frame Deletion,97 (0.25),0 (0.00)
